# Reproducibility and October-Readiness Evidence | Issue #14

Perform a clean reproducibility check of the September workflow and record the exact project, source, runtime, and artifact versions used for Gate 3.

This notebook:

- verifies the run begins from a clean committed project revision;
- reruns the final September notebooks;
- compares regenerated evidence against the frozen artifacts;
- records source, code, dependency, runtime, threshold, binning, and plot settings;
- creates Workflow Run and Artifact Bundle IDs;
- drafts the October evaluation plan;
- records final approval, disagreements, and limitations.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import os
import platform
import re
import shutil
import subprocess
import uuid
import tempfile

import numpy as np
import pandas as pd


def find_repo_root():
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError("Repository root not found.")


ROOT = find_repo_root()

SEPTEMBER = (
    ROOT
    / "notebooks"
    / "final-deliverables"
    / "September"
)

GATE1 = SEPTEMBER / "Gate 1"
GATE2 = SEPTEMBER / "Gate 2"
GATE3 = SEPTEMBER / "Gate 3"

OUTPUT_DIR = (
    GATE3
    / "reproducibility-and-october-readiness"
)

RERUN_DIR = OUTPUT_DIR / "rerun_notebooks"

# Temporary baseline files should not live inside the repository.
# Use short, flat filenames to avoid Windows path-length issues.
SNAPSHOT_DIR = Path(
    tempfile.mkdtemp(prefix="allstate_repro_")
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RERUN_DIR.mkdir(parents=True, exist_ok=True)

print("Temporary snapshot directory:", SNAPSHOT_DIR)

SOURCE_PATH = ROOT / "data" / "allstate_claims_data.csv"

ROUNDING_DECIMALS = 6
NUMERIC_ATOL = 10 ** (-ROUNDING_DECIMALS)

RARE_LEVEL_THRESHOLD = 188
REQUESTED_CONTINUOUS_BINS = 10
TARGET_HISTOGRAM_BINS = 100

print("Repository:", ROOT)
print("Output:", OUTPUT_DIR.relative_to(ROOT))

Temporary snapshot directory: /var/folders/b4/mzx1tj753sdgm4pjdk68j9gr0000gn/T/allstate_repro_9hgc_ifh
Repository: /Users/ragibn/Library/CloudStorage/OneDrive-Personal/Documents/Programs/BTT - AI Studio Fall/Allstate-1A-predicting-auto-claims-severity
Output: notebooks/final-deliverables/September/Gate 3/reproducibility-and-october-readiness


## 1. Record Clean Project and Source Identity

The reproducibility run must begin from a committed project revision.

Record:

- Git branch and commit;
- whether the working tree was clean at run start;
- source-data SHA-256;
- source file size;
- Python and notebook/runtime versions; and
- Workflow Run ID.

In [2]:
def git(*args):
    return subprocess.check_output(
        ["git", *args],
        cwd=ROOT,
        text=True
    ).strip()


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


GIT_COMMIT = git("rev-parse", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_STATUS_AT_START = git("status", "--porcelain")
STARTED_CLEAN = GIT_STATUS_AT_START == ""

if not STARTED_CLEAN:
    raise RuntimeError(
        "Final reproducibility run must begin from a clean committed "
        "working tree. Commit/stash changes and rerun."
    )

SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_SIZE = SOURCE_PATH.stat().st_size

UTC_START = datetime.now(timezone.utc)

WORKFLOW_RUN_ID = (
    f"G3-{UTC_START.strftime('%Y%m%dT%H%M%SZ')}-"
    f"{GIT_COMMIT[:8]}"
)

try:
    import google.colab  # noqa
    RUNTIME_CONTEXT = "Google Colab"
except ImportError:
    RUNTIME_CONTEXT = "VS Code / local Jupyter"

run_identity = {
    "workflow_run_id": WORKFLOW_RUN_ID,
    "run_started_utc": UTC_START.isoformat(),
    "git_branch": GIT_BRANCH,
    "git_commit": GIT_COMMIT,
    "working_tree_clean_at_start": STARTED_CLEAN,
    "source_path": SOURCE_PATH.relative_to(ROOT).as_posix(),
    "source_sha256": SOURCE_SHA256,
    "source_size_bytes": SOURCE_SIZE,
    "runtime_context": RUNTIME_CONTEXT,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
}

display(pd.DataFrame([run_identity]))

,workflow_run_id,run_started_utc,git_branch,git_commit,working_tree_clean_at_start,source_path,source_sha256,source_size_bytes,runtime_context,python_version,platform
0,G3-20261006T061322Z-157d5e6c,2026-10-06T06:13:22.754890+00:00,main,157d5e6cac0e0eaae1e0948a6818e53c316b991d,True,data/allstate_claims_data.csv,74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf...,70025339,VS Code / local Jupyter,3.14.2,macOS-27.0-arm64-arm-64bit-Mach-O


## 2. Record Dependencies and Analysis Settings

Freeze the environment and the analysis settings required by Issue #14.

The recorded settings include:

- dependency versions;
- random-seed declarations found in the final notebooks;
- target histogram binning;
- categorical rare-level threshold;
- requested continuous quantile bins; and
- numerical reproducibility precision.

In [3]:
PACKAGES = [
    "numpy",
    "pandas",
    "matplotlib",
    "scipy",
    "scikit-learn",
    "jupyter",
    "nbformat",
    "nbclient",
]

dependency_rows = []

for package in PACKAGES:
    try:
        version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        version = "not installed"

    dependency_rows.append({
        "package": package,
        "version": version,
    })

dependencies = pd.DataFrame(dependency_rows)

dependencies.to_csv(
    OUTPUT_DIR / "dependency_versions.csv",
    index=False
)

settings = {
    "target_histogram_bins": TARGET_HISTOGRAM_BINS,
    "categorical_rare_level_threshold": RARE_LEVEL_THRESHOLD,
    "continuous_requested_quantile_bins": REQUESTED_CONTINUOUS_BINS,
    "numeric_comparison_rounding_decimals": ROUNDING_DECIMALS,
    "numeric_absolute_tolerance": NUMERIC_ATOL,
    "categorical_support_rule": (
        "support < 188 is rare for EDA display/support; "
        "not an automatic removal rule"
    ),
    "continuous_bin_rule": (
        "request 10 quantile bins; record fewer bins when repeated "
        "values prevent distinct boundaries"
    ),
}

with open(OUTPUT_DIR / "analysis_settings.json", "w") as f:
    json.dump(settings, f, indent=2)

display(dependencies)
display(pd.DataFrame([settings]))

,package,version
0,numpy,2.5.1
1,pandas,3.0.5
2,matplotlib,3.11.2
3,scipy,1.18.1
4,scikit-learn,not installed
5,jupyter,not installed
6,nbformat,5.11.1
7,nbclient,0.11.0


,target_histogram_bins,categorical_rare_level_threshold,continuous_requested_quantile_bins,numeric_comparison_rounding_decimals,numeric_absolute_tolerance,categorical_support_rule,continuous_bin_rule
0,100,188,10,6,0.000001,support < 188 is rare for EDA display/support;...,request 10 quantile bins; record fewer bins wh...


In [4]:
NOTEBOOKS_TO_RUN = [
    GATE1 / "task_1.ipynb",
    GATE1 / "task_2.ipynb",
    GATE2 / "target_analysis.ipynb",
    GATE2 / "categorical_analysis_method.ipynb",
    GATE2 / "continuous_analysis.ipynb",
    GATE2 / "findings_and_review.ipynb",
]

for path in NOTEBOOKS_TO_RUN:
    assert path.is_file(), f"Missing notebook: {path}"


seed_patterns = [
    r"random_state\s*=\s*[^,\)\n]+",
    r"np\.random\.seed\([^\)]*\)",
    r"random\.seed\([^\)]*\)",
    r"tf\.random\.set_seed\([^\)]*\)",
]

seed_rows = []

for notebook_path in NOTEBOOKS_TO_RUN:
    text = notebook_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    matches = []

    for pattern in seed_patterns:
        matches.extend(re.findall(pattern, text))

    seed_rows.append({
        "notebook": notebook_path.relative_to(ROOT).as_posix(),
        "seed_declarations": " | ".join(sorted(set(matches)))
        if matches else "none found / not required",
    })

seed_inventory = pd.DataFrame(seed_rows)

seed_inventory.to_csv(
    OUTPUT_DIR / "random_seed_inventory.csv",
    index=False
)

display(seed_inventory)

,notebook,seed_declarations
0,notebooks/final-deliverables/September/Gate 1/...,none found / not required
1,notebooks/final-deliverables/September/Gate 1/...,none found / not required
2,notebooks/final-deliverables/September/Gate 2/...,none found / not required
3,notebooks/final-deliverables/September/Gate 2/...,none found / not required
4,notebooks/final-deliverables/September/Gate 2/...,none found / not required
5,notebooks/final-deliverables/September/Gate 2/...,none found / not required


## 3. Freeze the Pre-Run Artifact State

Snapshot the final September evidence before rerunning the workflow.

CSV files are later compared cell-by-cell:

- non-numeric values must match exactly;
- numeric values must match after rounding to 6 decimal places.

JSON files are compared semantically.

Figure filenames and hashes are also recorded.

In [5]:
ARTIFACT_EXTENSIONS = {
    ".csv",
    ".json",
    ".png",
    ".jpg",
    ".jpeg",
    ".svg",
}


def collect_artifacts():
    rows = []

    for gate in [GATE1, GATE2, GATE3]:
        for path in gate.rglob("*"):

            if not path.is_file():
                continue

            # Do not include this reproducibility task's own outputs.
            if OUTPUT_DIR in path.parents:
                continue

            if path.suffix.lower() not in ARTIFACT_EXTENSIONS:
                continue

            rows.append({
                "gate": gate.name,
                "path": path.relative_to(ROOT).as_posix(),
                "extension": path.suffix.lower(),
                "sha256": sha256_file(path),
            })

    return (
        pd.DataFrame(rows)
        .sort_values("path")
        .reset_index(drop=True)
    )


def snapshot_path(rel_path):
    """
    Give every snapshot a short unique filename.

    The original repository path is still stored in artifact_before,
    so no provenance is lost.
    """
    rel_path = str(rel_path)

    short_id = hashlib.sha256(
        rel_path.encode("utf-8")
    ).hexdigest()[:16]

    suffix = Path(rel_path).suffix

    return SNAPSHOT_DIR / f"{short_id}{suffix}"


artifact_before = collect_artifacts()

snapshot_manifest_rows = []

for rel_path in artifact_before["path"]:

    source = ROOT / rel_path
    destination = snapshot_path(rel_path)

    shutil.copy2(source, destination)

    snapshot_manifest_rows.append({
        "artifact": rel_path,
        "snapshot_file": destination.name,
        "baseline_sha256": sha256_file(source),
    })


snapshot_manifest = pd.DataFrame(snapshot_manifest_rows)

print(f"Snapshotted {len(artifact_before)} artifacts.")
print("Temporary snapshots are outside the repository.")

display(artifact_before)

Snapshotted 72 artifacts.
Temporary snapshots are outside the repository.


,gate,path,extension,sha256
0,Gate 1,notebooks/final-deliverables/September/Gate 1/...,.csv,14ef91d2322836a4de9158baeb0c499a1799bd7bc1bfa2...
1,Gate 1,notebooks/final-deliverables/September/Gate 1/...,.csv,9f67f435eea2076801cc9f3faab925bc531ea5b07cf57e...
2,Gate 1,notebooks/final-deliverables/September/Gate 1/...,.csv,3d41a6c1fbef315f962c88846aea07cf158349e86cd5a5...
3,Gate 2,notebooks/final-deliverables/September/Gate 2/...,.csv,1a86cb04eed046f115b5d3748be1f93f26aacafdfb2331...
4,Gate 2,notebooks/final-deliverables/September/Gate 2/...,.csv,f25fd20cd7394785afc014483b5c9c4cad755baa759ad4...
...,...,...,...,...
67,Gate 3,notebooks/final-deliverables/September/Gate 3/...,.csv,53c20e7c2302ec447be3cfed7e33dc293cfc19b5662964...
68,Gate 3,notebooks/final-deliverables/September/Gate 3/...,.csv,52827480f708cea6aa18dd4b094422c29f2d72ed8eeea0...
69,Gate 3,notebooks/final-deliverables/September/Gate 3/...,.json,2826a7a1d11722bfd1a48550d0860533d731dcfdc532ee...
70,Gate 3,notebooks/final-deliverables/September/Gate 3/...,.csv,774a05fa959fc8ae0c755feb9b429fd7d1e16d3ccb766c...


## 4. Rerun the Final September Workflow

Execute the final Gate 1 and Gate 2 notebooks in workflow order.

Executed notebook copies are saved under this Gate 3 evidence folder.

The notebooks still read and write through the repository paths used by the final workflow, allowing regenerated evidence to be compared with the frozen artifacts.

In [6]:
import nbformat
from nbclient.client import NotebookClient


rerun_results = []

original_cwd = Path.cwd()

try:
    for notebook_path in NOTEBOOKS_TO_RUN:

        print(f"Running {notebook_path.relative_to(ROOT)} ...")

        notebook = nbformat.read(
            notebook_path,
            as_version=4
        )

        kernel_name = (
            notebook.metadata
            .get("kernelspec", {})
            .get("name", "python3")
        )

        os.chdir(notebook_path.parent)

        client = NotebookClient(
            notebook,
            timeout=1200,
            kernel_name=kernel_name,
            allow_errors=False,
        )

        executed = client.execute()

        output_name = (
            notebook_path.parent.name.replace(" ", "_")
            + "__"
            + notebook_path.name
        )

        output_path = RERUN_DIR / output_name

        nbformat.write(executed, output_path)

        rerun_results.append({
            "notebook": notebook_path.relative_to(ROOT).as_posix(),
            "status": "PASS",
            "executed_copy": output_path.relative_to(ROOT).as_posix(),
        })

finally:
    os.chdir(original_cwd)


rerun_results = pd.DataFrame(rerun_results)

rerun_results.to_csv(
    OUTPUT_DIR / "notebook_rerun_results.csv",
    index=False
)

display(rerun_results)

assert len(rerun_results) == len(NOTEBOOKS_TO_RUN)
assert rerun_results["status"].eq("PASS").all()

print("September notebook rerun: PASS")

Running notebooks/final-deliverables/September/Gate 1/task_1.ipynb ...


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


Running notebooks/final-deliverables/September/Gate 1/task_2.ipynb ...


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


Running notebooks/final-deliverables/September/Gate 2/target_analysis.ipynb ...


...

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


Running notebooks/final-deliverables/September/Gate 2/findings_and_review.ipynb ...


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


,notebook,status,executed_copy
0,notebooks/final-deliverables/September/Gate 1/...,PASS,notebooks/final-deliverables/September/Gate 3/...
1,notebooks/final-deliverables/September/Gate 1/...,PASS,notebooks/final-deliverables/September/Gate 3/...
2,notebooks/final-deliverables/September/Gate 2/...,PASS,notebooks/final-deliverables/September/Gate 3/...
3,notebooks/final-deliverables/September/Gate 2/...,PASS,notebooks/final-deliverables/September/Gate 3/...
4,notebooks/final-deliverables/September/Gate 2/...,PASS,notebooks/final-deliverables/September/Gate 3/...
5,notebooks/final-deliverables/September/Gate 2/...,PASS,notebooks/final-deliverables/September/Gate 3/...


September notebook rerun: PASS


## 5. Compare Regenerated Evidence

Verify that the rerun reproduces the frozen evidence.

Rules:

- artifact filename inventory must remain unchanged;
- discrete CSV values must match exactly;
- numeric CSV values must match within the recorded six-decimal precision;
- JSON evidence must be semantically identical;
- figure names must remain present;
- hashes are recorded for all artifacts.

A failure is preserved as evidence and must be investigated rather than overridden.

In [7]:
artifact_after = collect_artifacts()

before_paths = set(artifact_before["path"])
after_paths = set(artifact_after["path"])

missing_after = sorted(before_paths - after_paths)
unexpected_after = sorted(after_paths - before_paths)

assert not missing_after, (
    f"Artifacts missing after rerun: {missing_after}"
)

assert not unexpected_after, (
    f"Unexpected artifacts after rerun: {unexpected_after}"
)


def compare_csv(baseline_path, current_path):
    old = pd.read_csv(baseline_path)
    new = pd.read_csv(current_path)

    if old.columns.tolist() != new.columns.tolist():
        return False, "column mismatch"

    if old.shape != new.shape:
        return False, "shape mismatch"

    for column in old.columns:

        old_col = old[column]
        new_col = new[column]

        old_numeric = pd.to_numeric(old_col, errors="coerce")
        new_numeric = pd.to_numeric(new_col, errors="coerce")

        numeric_mask = old_numeric.notna() & new_numeric.notna()

        # Numeric cells
        if numeric_mask.any():

            old_values = old_numeric[numeric_mask].round(
                ROUNDING_DECIMALS
            )

            new_values = new_numeric[numeric_mask].round(
                ROUNDING_DECIMALS
            )

            if not np.allclose(
                old_values,
                new_values,
                atol=NUMERIC_ATOL,
                rtol=0,
                equal_nan=True,
            ):
                return False, f"numeric mismatch in {column}"

        # Non-numeric / discrete cells
        discrete_mask = ~numeric_mask

        old_discrete = (
            old_col[discrete_mask]
            .fillna("<NA>")
            .astype(str)
        )

        new_discrete = (
            new_col[discrete_mask]
            .fillna("<NA>")
            .astype(str)
        )

        if not old_discrete.equals(new_discrete):
            return False, f"discrete mismatch in {column}"

    return True, "match"


comparison_rows = []

for rel_path in sorted(before_paths):

    baseline_path = snapshot_path(rel_path)
    current_path = ROOT / rel_path
    suffix = current_path.suffix.lower()

    if suffix == ".csv":

        passed, detail = compare_csv(
            baseline_path,
            current_path
        )

    elif suffix == ".json":

        with open(baseline_path) as f:
            old_json = json.load(f)

        with open(current_path) as f:
            new_json = json.load(f)

        passed = old_json == new_json
        detail = "semantic match" if passed else "JSON mismatch"

    else:
        # Figures: require the same artifact to exist.
        # Hash is recorded separately for review.
        passed = current_path.is_file()
        detail = "figure present"

    comparison_rows.append({
        "artifact": rel_path,
        "type": suffix,
        "status": "PASS" if passed else "FAIL",
        "detail": detail,
        "baseline_sha256": sha256_file(baseline_path),
        "rerun_sha256": sha256_file(current_path),
        "binary_hash_match": (
            sha256_file(baseline_path)
            == sha256_file(current_path)
        ),
    })


comparisons = pd.DataFrame(comparison_rows)

comparisons.to_csv(
    OUTPUT_DIR / "reproducibility_checks.csv",
    index=False
)

display(
    comparisons[
        comparisons["status"] != "PASS"
    ]
)

assert comparisons["status"].eq("PASS").all()

print("Artifact reproducibility checks: PASS")

,artifact,type,status,detail,baseline_sha256,rerun_sha256,binary_hash_match
1,notebooks/final-deliverables/September/Gate 1/...,.csv,FAIL,discrete mismatch in source_dtype,9f67f435eea2076801cc9f3faab925bc531ea5b07cf57e...,8c17961b4b226738e93cc0d39db856151ac08af4d32a4d...,False
5,notebooks/final-deliverables/September/Gate 2/...,.csv,FAIL,numeric mismatch in bytes,22ddb5c1325301f175c6aa370c21b13dec82a0b2df0eb3...,8e27339ce5f17306a28097a95b5c5d5b86adeb16f72f37...,False


AssertionError: 

## 6. Create Workflow and Artifact Bundle IDs

Record exact artifact collections used to pass Gate 3.

Artifact Bundle IDs are derived from the sorted paths and SHA-256 hashes for each Gate.

In [ ]:
def bundle_id(gate_name, inventory):
    subset = (
        inventory[inventory["gate"] == gate_name]
        [["path", "sha256"]]
        .sort_values("path")
    )

    payload = "\n".join(
        f"{row.path}:{row.sha256}"
        for row in subset.itertuples()
    )

    digest = hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:16]

    return f"{gate_name.replace(' ', '').upper()}-{digest}"


BUNDLE_IDS = {
    gate: bundle_id(gate, artifact_after)
    for gate in ["Gate 1", "Gate 2", "Gate 3"]
}

artifact_manifest = artifact_after.copy()
artifact_manifest["workflow_run_id"] = WORKFLOW_RUN_ID
artifact_manifest["git_commit"] = GIT_COMMIT
artifact_manifest["source_sha256"] = SOURCE_SHA256

artifact_manifest.to_csv(
    OUTPUT_DIR / "artifact_manifest.csv",
    index=False
)

run_manifest = {
    **run_identity,
    "artifact_bundle_ids": BUNDLE_IDS,
    "rounding_decimals": ROUNDING_DECIMALS,
    "rare_level_threshold": RARE_LEVEL_THRESHOLD,
    "requested_continuous_bins": REQUESTED_CONTINUOUS_BINS,
    "target_histogram_bins": TARGET_HISTOGRAM_BINS,
}

with open(
    OUTPUT_DIR / "workflow_run_manifest.json",
    "w"
) as f:
    json.dump(run_manifest, f, indent=2)

print("Workflow Run ID:", WORKFLOW_RUN_ID)

for gate, bundle in BUNDLE_IDS.items():
    print(f"{gate} Artifact Bundle ID: {bundle}")

Workflow Run ID: G3-20260929T042604Z-5ece32b7
Gate 1 Artifact Bundle ID: GATE1-3c893f22f14ced97
Gate 2 Artifact Bundle ID: GATE2-8e529ab389c94c3d
Gate 3 Artifact Bundle ID: GATE3-c4860456366d67d3


## 7. October Evaluation Plan

Draft the October modeling/evaluation plan without tuning against validation results.

Requirements:

- split before fitting preprocessing;
- fit preprocessing only on training data;
- preserve an untouched validation partition;
- evaluate MAE in original `loss` units;
- include the Challenge Advisor's requested mean-constant baseline;
- also include a median-constant comparator;
- fit both constants using training data only;
- do not report full-dataset descriptive constant MAE as held-out performance;
- define rare and unseen-category handling without learning from validation data.

In [ ]:
october_plan = """# October Evaluation Plan

## Split and evaluation

1. Create the train/validation split before fitting any preprocessing.
2. Keep validation data untouched during preprocessing and model selection.
3. Fit all learned preprocessing steps using training data only.
4. Evaluate final validation predictions using MAE in original `loss` units.
5. Do not tune preprocessing or model choices against the final validation results.

## Required constant baselines

The first October comparison will include:

- **Mean-constant baseline:** predict the mean training-set `loss`.
- **Median-constant comparator:** predict the median training-set `loss`.

Both constants will be calculated from the training partition only.

The median comparator is included because the median minimizes absolute error for a constant prediction, while the Challenge Advisor specifically requested the mean-constant baseline.

No MAE calculated from a constant fitted to the complete dataset will be presented as held-out model performance.

## Preprocessing

All preprocessing that learns parameters from data must be fitted on the training partition only.

Examples include:

- category-frequency thresholds;
- categorical encoders;
- imputation rules;
- scaling or transformations;
- feature-selection thresholds; and
- any data-dependent bin boundaries.

## Rare and unseen categorical levels

Rare-level handling will be determined from training data only.

If a categorical encoder groups rare levels, the support threshold and grouping map will be learned from the training partition.

Previously unseen validation categories will be mapped to an explicit unknown/unseen representation supported by the encoder.

Validation category frequencies will not be used to change the training encoding map.

## Model experimentation

The constant baselines establish the initial comparison.

Subsequent models may compare feature-selection and modeling approaches, but the untouched validation partition will remain separate from preprocessing fitting and training.
"""

PLAN_PATH = OUTPUT_DIR / "october_evaluation_plan.md"

PLAN_PATH.write_text(
    october_plan,
    encoding="utf-8"
)

print(october_plan)

# October Evaluation Plan

## Split and evaluation

1. Create the train/validation split before fitting any preprocessing.
2. Keep validation data untouched during preprocessing and model selection.
3. Fit all learned preprocessing steps using training data only.
4. Evaluate final validation predictions using MAE in original `loss` units.
5. Do not tune preprocessing or model choices against the final validation results.

## Required constant baselines

The first October comparison will include:

- **Mean-constant baseline:** predict the mean training-set `loss`.
- **Median-constant comparator:** predict the median training-set `loss`.

Both constants will be calculated from the training partition only.

The median comparator is included because the median minimizes absolute error for a constant prediction, while the Challenge Advisor specifically requested the mean-constant baseline.

No MAE calculated from a constant fitted to the complete dataset will be presented as held-out model 

## 8. Team Approval and Final Gate 3 Sign-Off

Record the team's approval, disagreements, limitations, Workflow Run ID, and Artifact Bundle IDs.

Do not fill in approval on behalf of another teammate.

The final Google Colab confirmation should use the same committed project revision whenever possible.

In [ ]:
SIGNOFF_PATH = OUTPUT_DIR / "gate3_signoff.csv"

if not SIGNOFF_PATH.exists():

    signoff = pd.DataFrame([
        {
            "workflow_run_id": WORKFLOW_RUN_ID,
            "git_commit": GIT_COMMIT,
            "gate1_bundle_id": BUNDLE_IDS["Gate 1"],
            "gate2_bundle_id": BUNDLE_IDS["Gate 2"],
            "gate3_bundle_id": BUNDLE_IDS["Gate 3"],
            "runtime_context": RUNTIME_CONTEXT,
            "team_approval": "pending",
            "approved_by": "",
            "approval_date": "",
            "independent_reviewer": "",
            "review_date": "",
            "disagreements": "",
            "limitations": (
                "Google Colab confirmation is still required if this "
                "run was completed only in VS Code/local Jupyter."
            ),
            "review_notes": "",
        }
    ])

    signoff.to_csv(
        SIGNOFF_PATH,
        index=False
    )

else:
    signoff = pd.read_csv(
        SIGNOFF_PATH,
        dtype=str,
        keep_default_na=False
    )

display(signoff)

,workflow_run_id,git_commit,gate1_bundle_id,gate2_bundle_id,gate3_bundle_id,runtime_context,team_approval,approved_by,approval_date,independent_reviewer,review_date,disagreements,limitations,review_notes
0,G3-20260929T042604Z-5ece32b7,5ece32b745e57335eafc4dc95181475f551e17f0,GATE1-3c893f22f14ced97,GATE2-8e529ab389c94c3d,GATE3-c4860456366d67d3,VS Code / local Jupyter,pending,,,,,,Google Colab confirmation is still required if...,


## 9. Final Verification

This cell checks everything that can be verified automatically.

Human approval and the explicit Google Colab run remain separate recorded actions.

In [ ]:
signoff = pd.read_csv(
    SIGNOFF_PATH,
    dtype=str,
    keep_default_na=False
)

automatic_checks = {
    "clean_committed_start": STARTED_CLEAN,
    "all_workflow_notebooks_ran":
        rerun_results["status"].eq("PASS").all(),
    "all_artifact_comparisons_pass":
        comparisons["status"].eq("PASS").all(),
    "source_hash_recorded":
        bool(SOURCE_SHA256),
    "git_revision_recorded":
        bool(GIT_COMMIT),
    "dependencies_recorded":
        len(dependencies) > 0,
    "random_seed_inventory_recorded":
        len(seed_inventory) == len(NOTEBOOKS_TO_RUN),
    "analysis_settings_recorded":
        bool(settings),
    "workflow_run_id_recorded":
        bool(WORKFLOW_RUN_ID),
    "artifact_bundle_ids_recorded":
        all(BUNDLE_IDS.values()),
    "october_plan_created":
        PLAN_PATH.is_file(),
}

automatic_verification = pd.DataFrame([
    {
        "requirement": key,
        "status": "PASS" if value else "FAIL",
    }
    for key, value in automatic_checks.items()
])

automatic_verification.to_csv(
    OUTPUT_DIR / "automatic_verification.csv",
    index=False
)

display(automatic_verification)

assert all(automatic_checks.values())

COLAB_CONFIRMED = (
    RUNTIME_CONTEXT == "Google Colab"
)

TEAM_SIGNOFF_COMPLETE = (
    signoff.loc[0, "team_approval"] == "approved"
    and signoff.loc[0, "approved_by"] != ""
    and signoff.loc[0, "independent_reviewer"] != ""
)

print("\nAutomatic reproducibility checks: PASS")
print("Google Colab confirmation:", COLAB_CONFIRMED)
print("Team sign-off complete:", TEAM_SIGNOFF_COMPLETE)

if COLAB_CONFIRMED and TEAM_SIGNOFF_COMPLETE:
    print("\nIssue #14: READY TO CLOSE")
else:
    print("\nIssue #14: AUTOMATED WORK COMPLETE")
    print(
        "Complete the remaining Colab confirmation and/or "
        "team sign-off before closing."
    )

,requirement,status
0,clean_committed_start,PASS
1,all_workflow_notebooks_ran,PASS
2,all_artifact_comparisons_pass,PASS
3,source_hash_recorded,PASS
4,git_revision_recorded,PASS
5,dependencies_recorded,PASS
6,random_seed_inventory_recorded,PASS
7,analysis_settings_recorded,PASS
8,workflow_run_id_recorded,PASS
9,artifact_bundle_ids_recorded,PASS



Automatic reproducibility checks: PASS
Google Colab confirmation: False
Team sign-off complete: False

Issue #14: AUTOMATED WORK COMPLETE
Complete the remaining Colab confirmation and/or team sign-off before closing.
